## 라이브러리 호출 및 데이터 로드

In [93]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json
from pprint import pprint

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

### 🔽데이터 로드 경로 변경하셔야 합니다!

- 파일경로 가장 앞 부분은 `../../`로 변경하시면 csv가 저장된 위치에서 내부 개인작업 폴더로 로드 가능하십니다.
- 1차 파생테이블에서 파생컬럼을 새롭게 만드시는 경우, 의미도 함께 마크다운 or 주석으로 남겨주세요..

In [94]:
# 데이터 로드
# user 단위 match 데이터(전처리 2차 완료)
df_meta_2 = pd.read_csv('../preprocess_trial_csv/유저단위_게임데이터_상위랭커보존-2차 전처리 완료(match).csv')

## User-level 데이터 재구성 사유 (요약)

* `explode` 과정에서 하나의 관측치가 여러 행으로 분할되어 **표본 수가 과대 증가**
* 동일 게임/유저 기반 데이터가 반복되며 **독립성 가정 위반**
* 이로 인해 **통계 검정 결과(p-value)가 왜곡될 가능성 존재**

따라서, User-level 데이터는 `explode` 없이 원본 구조를 유지하고,
분석은 **Game-level 기준으로 수행**하여 신뢰도를 확보함.


In [95]:
df_meta_2.head(2)

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,user_tier,season,user_id,flag_1,flag_2,mix_flag,combination_rebuild,active_synergies,top4_flag,ranked_1
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum,season 3,KR-USER-1,0,0,normal,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Brawler': 1, 'Celestial': 1, 'Void': 1, 'Sniper': 1}",{},0,0
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum,season 3,KR-USER-2,0,0,normal,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Blademaster': 2, 'Brawler': 1, 'Sorcerer': 1, 'Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Cybernetic': 3, 'Vanguard': 2}",1,0


---
## champion 테이블 전처리

### champion 테이블 내 class 컬럼에 존재하는 오타 수정
- `Infiltrato` → `Infiltrator` 오타 수정

In [96]:
# champion 데이터 로드
champions = pd.read_csv('../raw_data/TFT_Champion_CurrentVersion.csv')

In [97]:
# Infiltrator 관련 전체 조회 (오타 포함)
infiltrator_all = champions[
    champions['class'].str.contains('Infiltrat', case=False, na=False)
][['name', 'class']]

print(f"Infiltrat* 포함 Champion 개수: {len(infiltrator_all)}")
print(infiltrator_all)

Infiltrat* 포함 Champion 개수: 5
      name            class
18   shaco  ['Infiltrator']
30    ekko   ['Infiltrato']
45   kaisa  ['Infiltrator']
46  khazix  ['Infiltrator']
51    fizz  ['Infiltrator']


In [98]:
# 정상 값만 출력(Infiltrator)
infiltrator_correct = champions[
    champions['class'].str.contains(r'\bInfiltrator\b', case=False, na=False)
][['name', 'class']]
print(f"\n정상 Infiltrator 개수: {len(infiltrator_correct)}")


정상 Infiltrator 개수: 4


In [99]:
# 오타만 따로 추출 (정상값 제외)
infiltrator_typo = infiltrator_all[
    ~infiltrator_all['class'].str.contains(r'\bInfiltrator\b', case=False, na=False)
]

print(f"\n오타 의심 항목")
print(infiltrator_typo)


오타 의심 항목
    name           class
30  ekko  ['Infiltrato']


In [100]:
# 오타 수정
champions['class'] = champions['class'].str.replace(
    r'\bInfiltrato\b', 'Infiltrator', regex=True
)

In [101]:
# 수정 후 확인
infiltrator_after = champions[champions['class'].str.contains('Infiltrator', na=False)][['name', 'class']]
print(f"Infiltrator 포함 Champion 개수: {len(infiltrator_after)}")
print(infiltrator_after)

Infiltrator 포함 Champion 개수: 5
      name            class
18   shaco  ['Infiltrator']
30    ekko  ['Infiltrator']
45   kaisa  ['Infiltrator']
46  khazix  ['Infiltrator']
51    fizz  ['Infiltrator']


### 컬럼명 및 해당값 소문자로 변환

In [102]:
# 컬럼명 소문자 통일
champions.columns = champions.columns.str.lower()

champions.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             52 non-null     str    
 1   cost             52 non-null     int64  
 2   health           52 non-null     int64  
 3   defense          52 non-null     int64  
 4   attack           52 non-null     int64  
 5   attack_range     52 non-null     int64  
 6   speed_of_attack  52 non-null     float64
 7   dps              52 non-null     int64  
 8   skill_name       52 non-null     str    
 9   skill_cost       52 non-null     str    
 10  origin           52 non-null     str    
 11  class            52 non-null     str    
dtypes: float64(1), int64(6), str(5)
memory usage: 5.0 KB


In [103]:
champions

,name,cost,health,defense,attack,attack_range,speed_of_attack,dps,skill_name,skill_cost,origin,class
0,gangplank,5,1000,30,60,1,1.00,60,gangplank_orbitalstrike,100/175,Space Pirate,"['Mercenary', 'Demolitionist']"
1,graves,1,650,35,55,1,0.55,30,graves_smokegrenade,50/80,Space Pirate,['Blaster']
2,neeko,3,800,35,50,2,0.65,33,neeko_popblossom,75/150,Star Guardian,['Protector']
3,darius,2,750,35,60,1,0.65,39,darius_spacepirateguillotine,0/60,Space Pirate,['Mana-Reaver']
4,rakan,2,600,35,45,2,0.70,32,rakan_grandentrance,50/100,Celestial,['Protector']
5,lux,3,600,20,40,4,0.70,28,lux_luridbinding,50/100,Dark Star,['Sorcerer']
6,rumble,3,800,35,50,1,0.70,35,rumble_flamespitter,0/60,Mech-Pilot,['Demolitionist']
7,leona,1,600,40,50,1,0.55,28,leona_cyberbarrier,50/100,Cybernetic,['Vanguard']
8,lucian,2,500,25,50,4,0.70,35,lucian_relentlesspersuit,0/35,Cybernetic,['Blaster']
9,lulu,5,800,25,45,3,0.80,36,lulu_masspolymorph,75/150,Celestial,['Mystic']


### 새로운 데이프레임 제작
- match 데이터에서 챔피언과 관련된 정보만 담은 데이터 프레임 제작 (`df_meta_2`)

In [104]:
#데이터 type 딕셔너리로 변환

# combination 컬럼 형변환
df_meta_2['combination'] = df_meta_2['combination'].apply(ast.literal_eval)
# champion 컬럼 형변환
df_meta_2['champion'] = df_meta_2['champion'].apply(ast.literal_eval)
# combination_rebuild 컬럼 형변환
df_meta_2['combination_rebuild'] = df_meta_2['combination_rebuild'].apply(ast.literal_eval)
#  active_synergies 컬럼 형변환
df_meta_2['active_synergies'] = df_meta_2['active_synergies'].apply(ast.literal_eval)

In [105]:
type(df_meta_2['champion'].iloc[0])

dict

In [106]:
df_meta_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 396239 entries, 0 to 396238
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gameid               396239 non-null  str    
 1   gameduration         396239 non-null  float64
 2   level                396239 non-null  int64  
 3   lastround            396239 non-null  int64  
 4   ranked               396239 non-null  int64  
 5   ingameduration       396239 non-null  float64
 6   combination          396239 non-null  object 
 7   champion             396239 non-null  object 
 8   user_tier            396239 non-null  str    
 9   season               396239 non-null  str    
 10  user_id              396239 non-null  str    
 11  flag_1               396239 non-null  int64  
 12  flag_2               396239 non-null  int64  
 13  mix_flag             396239 non-null  str    
 14  combination_rebuild  396239 non-null  object 
 15  active_synergies     396239 

In [107]:
records = []

for _, row in df_meta_2.iterrows():
    
    base = {
        'game_id': row['gameid'],
        'user_tier': row['user_tier'],
        'ranked': row['ranked'],
        'user_id': row['user_id'],
        'flag_1': row['flag_1'],
        'flag_2': row['flag_2'],
        'active_synergies': row['active_synergies'],
        'top4_flag': row['top4_flag'],
        'ranked_1': row['ranked_1']
    }
    
    champ_dict = row['champion']
    
    for champ_name, champ_info in champ_dict.items():
        new_row = base.copy()
        
        new_row['champion_name'] = champ_name
        new_row['star'] = champ_info.get('star', None)
        
        records.append(new_row)

In [108]:
# 챔피언 펼친 DF
df_champ_expanded = pd.DataFrame(records)

#문자열 소문자 변환
df_champ_expanded['champion_name'] = df_champ_expanded['champion_name'].str.strip().str.lower()
champions['name'] = champions['name'].str.strip().str.lower()



# champions 테이블과 조인
df_joined = df_champ_expanded.merge(
    champions[['name', 'cost', 'origin', 'class']],
    left_on='champion_name',
    right_on='name',
    how='left'
).drop(columns=['name'])

In [109]:
# 다시 묶기
champion_with_details = (
    df_joined.groupby('user_id')
    .apply(lambda x: {
        'game_id': x['game_id'].iloc[0],
        'user_tier': x['user_tier'].iloc[0],
        'ranked': x['ranked'].iloc[0],
        'flag_1': x['flag_1'].iloc[0],
        'flag_2': x['flag_2'].iloc[0],
        'active_synergies': x['active_synergies'].iloc[0],
        'top4_flag': x['top4_flag'].iloc[0],
        'ranked_1': x['ranked_1'].iloc[0],
        
        'champions': [
            {
                'name': r['champion_name'],
                'star': r['star'],
                'cost': r['cost'],
                'origin': r['origin'],
                'class': r['class']
            }
            for _, r in x.iterrows()
        ]
    })
    .apply(pd.Series)
    .reset_index()
)

In [110]:
type(champion_with_details['champions'].iloc[0]) #list
champion_with_details.head(2)

# chamions 칼럼 구조 
# [
#   {'name': 'Ziggs', 'star': 1, 'cost': 1, 'origin': 'Rebel', 'class': 'Demolitionist'},
#   {'name': 'Ashe', 'star': 1, 'cost': 3, 'origin': 'Celestial', 'class': 'Sniper'}
# ]

,user_id,game_id,user_tier,ranked,flag_1,flag_2,active_synergies,top4_flag,ranked_1,champions
0,KR-USER-1,KR_4291707834,platinum,5,0,0,{},0,0,"[{'name': 'ziggs', 'star': 1, 'cost': 1, 'origin': 'Rebel', 'class': '['Demolitionist']'}, {'name': 'ashe', 'star': 1, 'cost': 3, 'origin': 'Celestial', 'class': '['Sniper']'}, {'name': 'chogath', 'star': 1, 'cost': 4, 'origin': 'Void', 'class': '['Brawler']'}, {'name': 'ekko', 'star': 1, 'cost': 5, 'origin': 'Cybernetic', 'class': '['Infiltrator']'}]"
1,KR-USER-10,KR_4291614366,platinum,2,0,0,"{'Chrono': 4, 'DarkStar': 3, 'Brawler': 2, 'Sniper': 2}",1,0,"[{'name': 'malphite', 'star': 2, 'cost': 1, 'origin': 'Rebel', 'class': '['Brawler']'}, {'name': 'caitlyn', 'star': 2, 'cost': 1, 'origin': 'Chrono', 'class': '['Sniper']'}, {'name': 'blitzcrank', 'star': 2, 'cost': 2, 'origin': 'Chrono', 'class': '['Brawler']'}, {'name': 'shen', 'star': 2, 'cost': 2, 'origin': 'Chrono', 'class': '['Blademaster']'}, {'name': 'shaco', 'star': 2, 'cost': 3, 'origin': 'Dark Star', 'class': '['Infiltrator']'}, {'name': 'lux', 'star': 1, 'cost': 3, 'origin': 'Dark Star', 'class': '['Sorcerer']'}, {'name': 'wukong', 'star': 2, 'cost': 4, 'origin': 'Chrono', 'class': '['Vanguard']'}, {'name': 'jhin', 'star': 2, 'cost': 4, 'origin': 'Dark Star', 'class': '['Sniper']'}]"


###  champion statistic talbe 파일 csv 추출

In [111]:
# champion_with_details.to_csv("../preprocess_trial_csv/유저단위_게임데이터_상위랭커보존-stats_champion_1.csv", index=False, encoding='utf-8-sig')

## combination 정보가 담긴 새로운 데이터프레임 제작

In [112]:
# Combination 테이블 생성
cols = [
    'gameid',  
    'user_tier',
    'ranked',
    'user_id',
    'flag_1',
    'flag_2',
    'active_synergies',
    'top4_flag',
    'ranked_1',
    'combination_rebuild'
]

combination_table = df_meta_2[cols].copy()

In [113]:
combination_table.head(2)

,gameid,user_tier,ranked,user_id,flag_1,flag_2,active_synergies,top4_flag,ranked_1,combination_rebuild
0,KR_4291707834,platinum,5,KR-USER-1,0,0,{},0,0,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Brawler': 1, 'Celestial': 1, 'Void': 1, 'Sniper': 1}"
1,KR_4291707834,platinum,3,KR-USER-2,0,0,"{'Cybernetic': 3, 'Vanguard': 2}",1,0,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Blademaster': 2, 'Brawler': 1, 'Sorcerer': 1, 'Void': 1, 'Valkyrie': 1, 'Vanguard': 2}"


In [114]:
type(combination_table['combination_rebuild'].iloc[0])

dict

###  combination statistic talbe 파일 csv 추출

In [115]:
# combination_table.to_csv("../preprocess_trial_csv/유저단위_게임데이터_상위랭커보존-stats_combination_1.csv", index=False, encoding='utf-8-sig')

---
## items 테이블 전처리

### items 정보를 포함한 데이터프레임 제작

In [116]:
# item table의 item id에 대한 item의 영문 명칭 칼럼 조인 불가 - explocde로 붙이거나/ 딕셔너리가 더 복잡해질 가능성이 높음

# # 아이템 데이터 로드
# items_table = pd.read_csv("../raw_data/TFT_Item_CurrentVersion.csv")

In [117]:
def count_items(comb):
    if not isinstance(comb, dict):
        return 0
    
    total = 0
    for champ, info in comb.items():
        items = info.get('items', [])
        if isinstance(items, list):
            total += len(items)
    return total


stats_champion_item = df_meta_2[
    ['gameid', 'user_tier', 'ranked', 'champion']
].copy()

stats_champion_item['item_count'] = stats_champion_item['champion'].apply(count_items)
stats_champion_item['champion_count'] = stats_champion_item['champion'].apply(
    lambda x: len(x) if isinstance(x, dict) else 0
)

In [118]:
stats_champion_item.shape

(396239, 6)

In [119]:
stats_champion_item.head()

,gameid,user_tier,ranked,champion,item_count,champion_count
0,KR_4291707834,platinum,5,"{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",4,4
1,KR_4291707834,platinum,3,"{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",8,8
2,KR_4291707834,platinum,7,"{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",4,4
3,KR_4291707834,platinum,2,"{'Poppy': {'items': [], 'star': 2}, 'Xayah': {'items': [19, 23, 19], 'star': 3}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [16], 'star': 2}, 'Mordekaiser': {'items': [35, 67, 33], 'star': 3}, 'Ashe': {'items': [], 'star': 2}, 'Soraka': {'items': [68, 47], 'star': 2}}",9,7
4,KR_4291707834,platinum,1,"{'TwistedFate': {'items': [36, 27], 'star': 3}, 'Caitlyn': {'items': [49, 29], 'star': 2}, 'JarvanIV': {'items': [56], 'star': 2}, 'Blitzcrank': {'items': [15], 'star': 2}, 'Shen': {'items': [77, 6], 'star': 2}, 'Ezreal': {'items': [16], 'star': 2}, 'Lux': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 2}}",9,8


###  champion_itme statistic talbe 파일 csv 추출

In [120]:
# # csv 파일로 저장
# stats_champion_item.to_csv("../preprocess_trial_csv/유저단위_게임데이터_상위랭커보존-stats_champion_items_1.csv", index=False, encoding='utf-8-sig')